<a href="https://colab.research.google.com/github/HemanthSagarSR/AI_Lab-/blob/main/week4_CSS_Data_center_cooling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import scipy.special as special

class CoolingCuckooSearch:
    def __init__(self, server_heat_loads, max_safe_temp=75, ambient_temp=20,
                 num_nests=15, max_iter=100, pa=0.25):
        self.heat_loads = np.array(server_heat_loads)
        self.num_servers = len(server_heat_loads)
        self.max_safe_temp = max_safe_temp
        self.ambient_temp = ambient_temp
        self.num_nests = num_nests
        self.max_iter = max_iter
        self.pa = pa

        beta = 1.5
        sigma_num = special.gamma(1 + beta) * np.sin(np.pi * beta / 2)
        sigma_den = special.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2))
        self.sigma = (sigma_num / sigma_den) ** (1 / beta)
        self.beta = beta

    def _evaluate(self, nest):
        airflow = nest[:self.num_servers]
        temp_setting = nest[self.num_servers:]

        energy_consumption = np.sum((airflow ** 3) + (30 - temp_setting) * 2)

        estimated_server_temps = self.ambient_temp + (self.heat_loads / (airflow + 1e-5)) + (30 - temp_setting)

        penalty = 0
        for temp in estimated_server_temps:
            if temp > self.max_safe_temp:
                penalty += (temp - self.max_safe_temp) * 5000

        return energy_consumption + penalty

    def _levy_flight(self, nest):
        u = np.random.normal(0, self.sigma, self.num_servers * 2)
        v = np.random.normal(0, 1, self.num_servers * 2)
        step = u / (np.abs(v) ** (1 / self.beta))
        new_nest = nest + step * 0.2

        new_nest[:self.num_servers] = np.clip(new_nest[:self.num_servers], 1.0, 10.0)
        new_nest[self.num_servers:] = np.clip(new_nest[self.num_servers:], 18.0, 26.0)
        return new_nest

    def solve(self):
        low_bounds = [1.0] * self.num_servers + [18.0] * self.num_servers
        high_bounds = [10.0] * self.num_servers + [26.0] * self.num_servers

        nests = np.random.uniform(low_bounds, high_bounds, (self.num_nests, self.num_servers * 2))
        fitness = np.array([self._evaluate(n) for n in nests])

        best_nest = nests[np.argmin(fitness)].copy()
        best_score = np.min(fitness)

        for _ in range(self.max_iter):
            for i in range(self.num_nests):
                new_nest = self._levy_flight(nests[i])
                new_fit = self._evaluate(new_nest)

                if new_fit < fitness[i]:
                    nests[i] = new_nest
                    fitness[i] = new_fit

            for i in range(self.num_nests):
                if np.random.rand() < self.pa:
                    nests[i] = np.random.uniform(low_bounds, high_bounds, self.num_servers * 2)
                    fitness[i] = self._evaluate(nests[i])

            min_idx = np.argmin(fitness)
            if fitness[min_idx] < best_score:
                best_score = fitness[min_idx]
                best_nest = nests[min_idx].copy()

        return best_nest[:self.num_servers], best_nest[self.num_servers:], best_score

if __name__ == "__main__":
    num_zones = int(input("Enter number of server zones: "))
    print(f"Enter heat loads for the {num_zones} zones (space-separated in kW):")
    heat_inputs = [float(x) for x in input().split()]

    cs = CoolingCuckooSearch(heat_inputs)
    airflows, temps, energy = cs.solve()

    print("\n--- Optimization Complete ---")
    print("Optimized Airflow Rates (m^3/s):", np.round(airflows, 2))
    print("Optimized Temperature Settings (C):", np.round(temps, 2))
    print("Minimized Energy Cost Score:", round(energy, 2))


Enter number of server zones: 3
Enter heat loads for the 3 zones (space-separated in kW):
45.5 60.2 30.8

--- Optimization Complete ---
Optimized Airflow Rates (m^3/s): [1.55 2.69 1.  ]
Optimized Temperature Settings (C): [25.9  25.33 19.5 ]
Minimized Energy Cost Score: 62.66
